### Env Setup

In [2]:
import langchain
from dotenv import load_dotenv
import os


load_dotenv()
gemini_api_key= os.getenv("gemini_api-key")
groq_api_key= os.getenv("groq_api-key")
os.environ["GEMINI_API_KEY"]= os.getenv("gemini_api-key")
os.environ["GROQ_API_KEY"]= os.getenv("groq_api-key")

print('='*70)
print("groq_api_key:",groq_api_key[:10])
print('='*70)
print("gemini_api_key:",gemini_api_key[:10])


groq_api_key: gsk_lhjNVo
gemini_api_key: AIzaSyBMAz


### Getting started with Gemini Api

* how to invoke messages
* how to see reponse
* how to straeming

In [ ]:
os.environ["GEMINI_API_KEY"]= os.getenv("gemini_api-key")

# with model class

# from langchain_google_genai import ChatGoogleGenerativeAI

# model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")




# with __init_chat_model

from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")



# response = model.invoke("Explain RAG in simple terms")
# print(response.content)
from langchain_core.messages import SystemMessage,HumanMessage
messages=[
    SystemMessage("You are a helpful ai assistance,playful, so always response with emojis"),
    HumanMessage("what is benifit of langchain over other platforms?")
]

# response= model.invoke(messages)
# print(response)


# streaming the output messages
for chunk in model.stream(messages):
    print(chunk.content, end="\n", flush=True)




### Dynamic prompt tamplates



In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

os.environ["GEMINI_API_KEY"]= os.getenv("gemini_api-key")

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

# Creat a translation app


translation_tamplate= ChatPromptTemplate.from_messages([
    ("system","You are a professional translator.Transalate the following {message} from {source_lang} to {target_lang}"),
    ("user","{message}")
])

# using the above tamplate
prompt= translation_tamplate.invoke({
    "message":"Petrol price is growing up because of war.",
    "source_lang":"English",
    "target_lang":"Hindi"
})


prompt


ChatPromptValue(messages=[SystemMessage(content='You are a professional translator.Transalate the following Petrol price is growing up because of war. from English to Hindi', additional_kwargs={}, response_metadata={}), HumanMessage(content='Petrol price is growing up because of war.', additional_kwargs={}, response_metadata={})])

In [ ]:
translated_message= model.invoke(prompt)



content='युद्ध के कारण पेट्रोल की कीमतें बढ़ रही हैं।' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cb32a-e42e-7b63-9e3a-62fc1f46495d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 33, 'output_tokens': 10, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}


In [10]:
print(translated_message.content)

युद्ध के कारण पेट्रोल की कीमतें बढ़ रही हैं।


## Building Chains


In [26]:
import langchain
from dotenv import load_dotenv
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GEMINI_API_KEY"]= os.getenv("gemini_api-key")

model = init_chat_model("google_genai:gemini-2.5-flash-lite")



# def create_story_chain():
#     #template for story generation
#     story_prompt=ChatPromptTemplate([
#         ("system","You are a creative story teller. You have a good knownlwdge of {subject}. Write a youtube short ready story using the given Theme, charecter and setting."),
#         ("user","Theme:{theme} \n Main Charecter:{charecter} \n Setting:{setting}")
#     ])

#     #Tamplate for story Analysis
#     analysis_prompt=ChatPromptTemplate([
#         ("system","you are a food critc from the India. Analyze the folloeing story and give a brief insights."),
#         ("user","{text}")
#     ])

#     story_chain=(
#         story_prompt| model | StrOutputParser
#     )
#     analysis_chain=(
#         {"text":story_chain}
#         |analysis_prompt
#         |model
#         |StrOutputParser()

#     )
#     return analysis_chain



def create_story_chain():
    #template for story generation
    story_prompt=ChatPromptTemplate([
        ("system","You are a creative story teller. You have a good knownlwdge of food. Write a youtube short ready story using the given Theme, charecter and setting.Also make sure to use the language"),
        ("user","Theme:{theme} \n Main Charecter:{charecter} \n Setting:{setting} \n language:{lang}")
    ])

    #Tamplate for story Analysis
    analysis_prompt=ChatPromptTemplate([
        ("system","you are a food critc from the India. Analyze the folloeing story and give a brief insights."),
        ("user","{text}")
    ])

    story_chain=(
        story_prompt| model | StrOutputParser()
    )

    # function to pass a story for analysis in string format
    def analyze_story(story_text):
        return {"text": story_text}
    
    
    analysis_chain=(
        story_chain
        |RunnableLambda(analyze_story)
        |analysis_prompt
        |model
        |StrOutputParser()

    )
    return analysis_chain


In [16]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['charecter', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['theme'], input_types={}, partial_variables={}, template='You are a creative story teller. You have a good knownlwdge of {theme}. Write a youtube short ready story using the given Theme, charecter and setting.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['charecter', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme} \n Main Charecter:{charecter} \n Setting:{setting}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True,

In [27]:
chain = create_story_chain()

result = chain.invoke({
    "theme": "ancient",
    "charecter": "a curious foodie man",
    "setting": "an old city in 1990",
    "lang":"hindi"
})

print(result)

Namaste! As a food critic from India, this YouTube Short script, "The Taste of Time," truly resonates with the soul of our culinary heritage. Here are my brief insights:

**Nostalgia and Authenticity:** The script masterfully taps into the potent nostalgia for the 1990s, a period many Indians fondly remember as a time of simpler pleasures and unadulterated tastes. The grainy filter and retro music immediately transport the viewer, setting the stage for an authentic experience.

**The "Quest" Narrative:** Raj's "junooni khana khane wala" (passionate foodie) persona and his quest for a legendary sweet is a classic and engaging narrative. It highlights how food discovery was more about personal exploration and word-of-mouth before the digital age.

**The Unassuming Gem:** The focus on a small, unassuming sweet shop and an elderly shopkeeper embodies the spirit of countless hidden culinary treasures across India. These are the places that often hold the most authentic and time-tested recip